# Plant Leaf Disease Analysis - Experiment Notebook

This notebook uses the modular pipeline:
**Pre-processing -> Segmentation -> Morphological Ops -> Edge/Corner Detection -> Feature Extraction -> Classification**

In [ ]:
import glob
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np

from classifiers import train_and_evaluate
from pipeline import LeafAnalysisPipeline

In [ ]:
# Update this path to your local PlantVillage directory
dataset_root = r"path_to_PlantVillage"

pipeline = LeafAnalysisPipeline(
    preprocess_filter="gaussian",
    segmentation_method="otsu",
    morphology_operation="closing",
    edge_method="canny",
    corner_method="harris",
)

In [ ]:
# Pick one sample image for step-by-step visualization
sample_pattern = os.path.join(dataset_root, "*", "*.jpg")
sample_candidates = glob.glob(sample_pattern)

if len(sample_candidates) == 0:
    raise FileNotFoundError("No .jpg files found. Check dataset_root path.")

sample_image_path = sample_candidates[0]
results = pipeline.run_full_extractor(sample_image_path)

print("Sample image:", sample_image_path)
print("Feature vector length:", results["feature_vector"].shape[0])

In [ ]:
# Required 2x3 results grid for project reporting
figure, axes = plt.subplots(2, 3, figsize=(14, 9))

axes[0, 0].imshow(cv2.cvtColor(results["original"], cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("Original")
axes[0, 0].axis("off")

axes[0, 1].imshow(results["grayscale"], cmap="gray")
axes[0, 1].set_title("Grayscale")
axes[0, 1].axis("off")

axes[0, 2].imshow(results["mask"], cmap="gray")
axes[0, 2].set_title("Mask")
axes[0, 2].axis("off")

axes[1, 0].imshow(results["morphology"], cmap="gray")
axes[1, 0].set_title("Morphological Result")
axes[1, 0].axis("off")

axes[1, 1].imshow(results["edges"], cmap="gray")
axes[1, 1].set_title("Edges")
axes[1, 1].axis("off")

axes[1, 2].imshow(cv2.cvtColor(results["corners"], cv2.COLOR_BGR2RGB))
axes[1, 2].set_title("Corners")
axes[1, 2].axis("off")

figure.tight_layout()
plt.show()

In [ ]:
# Build a small experiment matrix (X, y) from dataset folders
# Folder structure expected: dataset_root/class_name/image.jpg
X = []
y = []

class_directories = [d for d in os.listdir(dataset_root) if os.path.isdir(os.path.join(dataset_root, d))]

for class_name in class_directories:
    class_folder = os.path.join(dataset_root, class_name)
    image_paths = glob.glob(os.path.join(class_folder, "*.jpg"))

    for image_path in image_paths:
        pipeline_output = pipeline.run_full_extractor(image_path)
        X.append(pipeline_output["feature_vector"])
        y.append(class_name)

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("Dataset matrix shape:", X.shape)
print("Labels shape:", y.shape)

In [ ]:
# Train and evaluate baseline classifiers
evaluation_results = train_and_evaluate(X, y)

for model_name, model_metrics in evaluation_results.items():
    print(f"Model: {model_name}")
    print(f"  Accuracy: {model_metrics['accuracy']:.4f}")
    print(f"  F1-score: {model_metrics['f1_score']:.4f}")
    model_metrics['confusion_matrix_figure'].show()
    print()

In [ ]:
# Placeholder: custom descriptor and Bag of Words experiments will be added later.